In [1]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.embeddings.cohere import CohereEmbedding
from llama_index.llms.groq import Groq
from llama_index.core import Settings

import os

## Settings

In [2]:

GROQ_API_KEY = os.environ["GROQ_API_KEY"] 
COHERE_API_KEY = os.environ["COHERE_API_KEY"] 
LLAMA_CLOUD_API_KEY = os.environ["LLAMA_CLOUD_API_KEY"]

llm = Groq(
    model="llama3-groq-70b-8192-tool-use-preview",
    api_key=GROQ_API_KEY
)

embed_model = CohereEmbedding(
    api_key=COHERE_API_KEY,
    model_name="embed-english-v3.0",
    input_type="search_query",
)


Settings.llm = llm 
Settings.embed_model = embed_model
# Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=20)
# Settings.num_output = 512
# Settings.context_window = 3900

In [3]:
# # Load documents and build index
# documents = SimpleDirectoryReader("../../data").load_data()
# index = VectorStoreIndex.from_documents(documents)

In [4]:

# # Query index
# query_engine = index.as_query_engine()
# response = query_engine.query("What is the paper about?")

# print(response)

## Reddis

```docker
docker run --name redis-vecdb -d -p 6379:6379 -p 8001:8001 redis/redis-stack:latest
```

In [5]:
from llama_index.core import SimpleDirectoryReader
# load documents
documents = SimpleDirectoryReader("../../data").load_data()
print(
    "Document ID:",
    documents[0].id_,
    "Document Filename:",
    documents[0].metadata["file_name"],
)

Document ID: 234d1dda-0742-4352-9aae-b39f353956a0 Document Filename: paper.pdf


In [2]:
from redisvl.schema import IndexSchema

custom_schema = IndexSchema.from_dict(
    {
        # customize basic index specs
        "index": {
            "name": "paper_index",
            "prefix": "paper",
            "key_separator": ":",
        },
        # customize fields that are indexed
        "fields": [
            # required fields for llamaindex
            {"type": "tag", "name": "id"},
            {"type": "tag", "name": "doc_id"},
            {"type": "text", "name": "text"},
            # custom metadata fields
            {"type": "numeric", "name": "updated_at"},
            {"type": "tag", "name": "file_name"},
            # custom vector field definition for embeddings
            {
                "type": "vector",
                "name": "vector",
                "attrs": {
                    "dims": 1024,  # updated dimension to match the embeddings
                    "algorithm": "hnsw",
                    "distance_metric": "cosine",
                },
            },
        ],
    }
)


In [3]:
custom_schema.to_yaml("db-schema.yaml")

In [7]:
custom_schema.index

IndexInfo(name='paper_index', prefix='paper', key_separator=':', storage_type=<StorageType.HASH: 'hash'>)

In [8]:
from llama_index.vector_stores.redis import RedisVectorStore
from llama_index.core import StorageContext, VectorStoreIndex
from redis import Redis

# create a Redis client connection
redis_client = Redis.from_url("redis://localhost:6379")

# create the vector store wrapper
vector_store = RedisVectorStore(
    schema=custom_schema,  # provide customized schema
    redis_client=redis_client,
    overwrite=True,
)

storage_context = StorageContext.from_defaults(vector_store=vector_store)

# build and load index from documents and storage context
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context
)

02:36:10 redisvl.index.index INFO   Index already exists, overwriting.


In [9]:
# Query index
query_engine = index.as_query_engine()
response = query_engine.query("What is the paper about?")

print(response)

The paper discusses the development of a new probabilistic method for identifying implicit feature mentions in customer reviews, which is crucial for high-recall applications in automated review analysis. It also explores the challenges of extracting key product features from product specifications and the importance of accurately identifying both explicit and implicit feature mentions in reviews.
